## Preliminares

In [1]:
import pandas as pd
import numpy as np
from src.config import data_folder
%load_ext autoreload
%autoreload 2
from src.transform import *

In [2]:
# Abrir archivo raw_data
df = pd.read_parquet(f"{data_folder}/raw_data.parquet")

# Se asegura el ordenamiento por fecha
df = df.sort_values(by='Date').reset_index(drop=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11372 entries, 0 to 11371
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11372 non-null  datetime64[ns]
 1   Ticker                       11372 non-null  object        
 2   Close                        11372 non-null  float64       
 3   Open                         11372 non-null  float64       
 4   Volume                       11372 non-null  float64       
 5   DateAdded                    6705 non-null   object        
 6   Sector                       11372 non-null  object        
 7   Industry                     11372 non-null  object        
 8   TotalRevenue                 11372 non-null  float64       
 9   GrossProfit                  10519 non-null  float64       
 10  OperatingIncome              11372 non-null  float64       
 11  NetIncome                    11372 non-nu

* Para mejorar la visualización de los datos, se expresan las columnas financieras y volumen en millones:

In [3]:
df = columnas_en_millones(df)

In [4]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,11372,2023-08-23 23:59:06.816743168,2020-09-01 00:00:00,2022-03-01 00:00:00,2023-09-01 00:00:00,2025-03-01 00:00:00,2026-06-01 00:00:00,NaN
Close,11372.0,1040.632239,0.57,41.010419,82.728504,165.380489,775000.0,22865.077155
Open,11372.0,1008.193167,0.53,40.525032,81.59261,161.329193,775648.0,22114.560395
Volume,11372.0,329.35621,0.0002,44.459425,103.586127,258.802775,36280.458,1251.880973
TotalRevenue,11372.0,6593.381045,-12134.0,1275.91075,2386.5,5278.8,227173.0,14899.692411
GrossProfit,10519.0,2361.834557,-24353.0,422.4545,831.622,1828.5,137192.0,6375.388808
OperatingIncome,11372.0,967.03061,-55512.0,125.3455,303.581,820.05,77648.0,2944.934098
NetIncome,11372.0,718.475912,-43621.0,65.47875,199.0,581.0,62578.0,2672.06322
EBITDA,1885.0,1561.779546,-16168.444,197.767,466.778,1202.0,84427.0,5209.528381
BasicAverageShares,11326.0,641.974123,0.00021,95.826629,219.801,550.57775,30864.0,1743.662701


## Corrección de errores y valores perdidos

In [5]:
# Se analiza el siguiente registro con valores extremadamente elevados en el estado de resultados
registro_iiin = df[(df['Ticker'] == 'IIIN') & (df['Date'] == '2021-12-01')]

# Mostrar el resultado
print(registro_iiin.T)

                                            2016
Date                         2021-12-01 00:00:00
Ticker                                      IIIN
Close                                   28.25313
Open                                   31.382216
Volume                                    7.3179
DateAdded                                   None
Sector                               Industrials
Industry                       Metal Fabrication
TotalRevenue                             171.258
GrossProfit                               39.919
OperatingIncome                           32.598
NetIncome                                 25.152
EBITDA                                       NaN
BasicAverageShares                        19.344
CashAndCashEquivalents                   89884.0
CurrentDebt                                  NaN
LongTermDebt                                 NaN
TotalDebt                                    NaN
StockholdersEquity                      302038.0
TotalAssets         

* Se observa una desconexión en los valores del Balance General, estan multiplicados por 1.000.
* El caso se detecto ya que arrojaba valores extremos en las métricas.

Se corrige dividiendo las columnas afectadas por 1.000:

In [ ]:
condicion_error_balance = (df['Ticker'] == 'IIIN') & (df['Date'] == '2021-12-01')

# Columnas del balance que tienen el error
columnas_error_balance = [
    'CashAndCashEquivalents', 
    'TotalAssets', 
    'StockholdersEquity', 
    'CurrentAssets', 
    'CurrentLiabilities'
]

# Se dividen por 1000
df.loc[condicion_error_balance, columnas_error_balance] /= 1000

Anomalías de signo: 

Las siguientes variables no pueden ser negativas:
* `TotalRevenue`
* `CurrentDebt`
* `LongTermDebt`
* `DepreciationAndAmortization`

In [ ]:
# Visualizar casos de TotalRevenue negativo
condicion_revenue_negativo = df['TotalRevenue'] < 0
cols_a_visualizar_revenue = ['Ticker', 'Date', 'TotalRevenue', 'OperatingIncome']
anomalias = df.loc[condicion_revenue_negativo, cols_a_visualizar_revenue]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

Empty DataFrame
Columns: [Ticker, Date, TotalRevenue, OperatingIncome]
Index: []
Cantidad de casos: 0


* No se puede concluir que se traten de alteraciones de signo en el parseo de datos de simFin. Se observa en 2 casos que el valor absoluto de `TotalRevenue` es mayor que `OperatingIncome`, es imposible que los resultados sean mayores que las ventas.

Se opta por asignar todos estos valores anómalos a NaN.

In [8]:
df['TotalRevenue'] = np.where(condicion_revenue_negativo, np.nan, df['TotalRevenue'])

* Casos de deuda negativa: se calcula "PasivoImplicito", que surge de aplicar la ecuación contable fundamental `Activo` = `Pasivo` + `Patrimonio Neto`

In [ ]:
# Calcular pasivo implícito
df['PasivoImplicito'] = df['TotalAssets'] - df['StockholdersEquity']

condicion_deuda_negativa = (df['CurrentDebt'] < 0) | (df['LongTermDebt'] < 0)
cols_a_visualizar_deuda = ['Ticker', 'Date', 'TotalDebt', 'CurrentDebt', 'LongTermDebt', 'PasivoImplicito']
anomalias = df.loc[condicion_deuda_negativa, cols_a_visualizar_deuda]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

Empty DataFrame
Columns: [Ticker, Date, TotalDebt, CurrentDebt, LongTermDebt, PasivoImplicito]
Index: []
Cantidad de casos: 0


* Se observa que la ecuación contable fundamental no se cumple. 

Se decide eliminar estos registros "tóxicos" del dataset.

In [11]:
df = df[~condicion_deuda_negativa].reset_index(drop=True)
df = df.drop(columns= 'PasivoImplicito')

* Casos de negativos en `DepreciationAndAmortization`: 

se analizan los casos considerando la ecuación `EBITDA` = `OperatingIncome` + `DepreciationAndAmortization`

In [ ]:
condicion_depre_negativa = df['DepreciationAndAmortization'] < 0
cols_a_visualizar_depre = ['Ticker', 'Date', 'DepreciationAndAmortization', 'EBITDA', 'OperatingIncome']
anomalias = df.loc[condicion_depre_negativa, cols_a_visualizar_depre]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

Empty DataFrame
Columns: [Ticker, Date, DepreciationAndAmortization, EBITDA, OperatingIncome]
Index: []
Cantidad de casos: 0


* Los datos que provienen de `simFin` utilizan signo negativo para esta columna, lo cual es incorrecto. 

* Dichos valores serán convertidos a positivos.

A continuación se verifican si hay datos negativos que vengan de yfinance (con EBITDA):

In [ ]:
condicion_depre_negativa_hay_ebitda = (df['DepreciationAndAmortization'] < 0) & (df['EBITDA'].notna())

anomalias = df.loc[condicion_depre_negativa_hay_ebitda, cols_a_visualizar_depre]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

Empty DataFrame
Columns: [Ticker, Date, DepreciationAndAmortization, EBITDA, OperatingIncome]
Index: []
Cantidad de casos: 0


* En este caso se observa que el valor implicito de `DepreciationAndAmortization` es igual a cero (EBITDA = OperatingIncome). 
* El valor de -0.3 pudo haberse tratado de un pequeño "ajuste contable", $300.000 dólares para una compañia del tamaño de `GameStop` es contablemente irrelevante.

Se reemplaza dicho valor por cero:

In [ ]:
df.loc[condicion_depre_negativa_hay_ebitda,'DepreciationAndAmortization'] = 0

In [ ]:
# Ahora se convierten a positivos los restantes
df['DepreciationAndAmortization'] = df['DepreciationAndAmortization'].abs()

# Verificar cambios
anomalias = df.loc[condicion_depre_negativa, cols_a_visualizar_depre]
print(anomalias)
print("Cantidad de casos:", len(anomalias))

Empty DataFrame
Columns: [Ticker, Date, DepreciationAndAmortization, EBITDA, OperatingIncome]
Index: []
Cantidad de casos: 0


In [ ]:
df.describe().T

In [ ]:
# Incidencia de missings
mostrar_missings(df)

* Se imputan parte de los NaNs en Variables Financieras antes de calcular métricas, 
mediante las relaciones contables entre ellas:

In [ ]:
df_acc_imputed = imputar_equivalencias_financieras(df)
mostrar_missings(df_acc_imputed)

* Se crea feature `YearsSinceAdded`:

In [ ]:
#  Pasar DateAdded a formato datetime, los NaN se vuelven NaT (not a time)
df_acc_imputed['DateAdded'] = pd.to_datetime(df_acc_imputed['DateAdded'], errors='coerce')
# Convertir a YearsSinceAdded, aqui los nulos regresan a NaN
df_acc_imputed['YearsSinceAdded'] = round(((pd.Timestamp.now() - df_acc_imputed['DateAdded']).dt.days / 365.25), 0)

# 3. Se asignan a cero años los tickers que no pertenecen al Índice S&P 500
df_acc_imputed['YearsSinceAdded'] = df_acc_imputed['YearsSinceAdded'].fillna(0)

# Eliminar la columna original
df_acc_imputed.drop('DateAdded', axis=1, inplace=True)
mostrar_missings(df_acc_imputed)

In [ ]:
# Se imputan las columnas financieras originales, por su media o mediana móvil según sus asimetrías
df_fin_imputed = imputar_numericas(df_acc_imputed)
mostrar_missings(df_fin_imputed)

* Se aplica forward fill y back fill para cubrir posibles huecos (es necesario que no hayan valores perdidos antes de calcular los valores TTM):

In [ ]:
df_fin_imputed = aplicar_fill(df_fin_imputed, limite=None)
mostrar_missings(df_fin_imputed)

* Se convierten las variables de flujo trimestrales a valores TTM (ventana móvil de 4 trimestres):

In [ ]:
df_ttm = transformar_flujos_a_ttm(df_fin_imputed)
mostrar_missings(df_ttm)

* Calcular métricas:

In [ ]:
df_with_metrics, crecimiento_cols = calcular_metricas(df_ttm)
mostrar_missings(df_with_metrics)

* Se aplica imputación transversal para las columnas de crecimiento:

In [ ]:
df_with_metrics = imputar_transversal(df_with_metrics, crecimiento_cols)
mostrar_missings(df_with_metrics)

* Calcular los retornos trimestrales, varianza del activo y covarianza con el mercado para cada ticker:

In [ ]:
# Se abre el fichero de precios del Índice del Mercado
df_index = pd.read_parquet(f"{data_folder}/market_index.parquet")
df_with_features = calcular_retornos(df_with_metrics, df_index)
mostrar_missings(df_with_features)

In [ ]:
df_with_features.describe().T

In [ ]:
# Analizar extremos
df_min = df_with_features.loc[df_with_features['EnterpriseValue'].idxmin()]
df_min

In [ ]:
# Se aplica lag de 1 período a volumen para evitar Data Leakage
df_with_features['Volume_Lag1'] = df_with_features['Volume'].shift(1)
df_with_features.drop('Volume', axis=1, inplace=True)

Imputación final de Valores Perdidos

In [ ]:
# Se aplica la imputación numérica sobre las nuevas variables
df_imputed = imputar_numericas(df_with_features)
mostrar_missings(df_imputed)

In [ ]:
# Se aplican los fills sobre los missings que queden
df_imputed = aplicar_fill(df_imputed,None)
mostrar_missings(df_imputed)

## Transformaciones

In [ ]:
# Se calculan tamaños relativos: RelativeAssets y RelativeRevenue
df_transformed = calcular_relative_size(df_imputed)
df_transformed.info()

In [ ]:
# Convertir Sector y Industry a tipo category
df_transformed['Sector'] = df_transformed['Sector'].astype('category')
df_transformed['Industry'] = df_transformed['Industry'].astype('category')

# Valores unicos en Sector
df_transformed['Sector'].value_counts()

In [ ]:
# Valores unicos en Industry
df_transformed['Industry'].value_counts()

In [ ]:
# Distribución de variables contínuas
df_transformed.describe().round(4).T

In [ ]:
# Coeficientes de asimetría
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Graficar
columna_a_graficar = 'EnterpriseValue' # indicar columna para el grafico
plot(df_transformed[columna_a_graficar])

In [ ]:
# Transformaciones logarítmicas

columnas_a_transformar = [ 
    'Volume_Lag1',
    'CapExToRevenue',
    'DebtToEquity',
    'QuarterlyVariance_Lag1',
    'MarketCap',
    'EnterpriseValue'
    ]
for columna in columnas_a_transformar:
    df_transformed[columna] = df_transformed[columna].fillna(0)
    df_transformed[f'{columna}_Log1p'] = np.log1p(df_transformed[columna])
    df_transformed.drop(columna, axis=1, inplace=True)

# Coeficientes de asimetria actualizado
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Definir columnas que saltean la "winsorización"
cols_fin_clean = obtener_cols_financieras(incluirTTM=True)

columnas_intactas = cols_fin_clean + [
    # Variables de precio y ratios
    'Close',
    'Open',    
    'TrailingPE',
    'EnterpriseToEbitda',
    'PriceToBook',
    # Otras
    'Date', 
    'Ticker'   
    ]

# Separar el dataset
df_passthrough = df_transformed[columnas_intactas].copy()
df_transformed_features = df_transformed.drop(columns=columnas_intactas)

## Gestión de Outliers

Se winsorizan los valores atipicos en las variables continuas que cumplan los siguientes criterios:

Para variables simetricas:
* A mas de 3 desviaciones tipicas de la media.
* Mas de 3 rangos intercuartilicos.

Para variables asimetricas (modulo del coeficiente de asimetrica mayor a 1):
* A mas de 3 MADs de la mediana.
* Mas de 3 rangos intercuartilicos.

In [ ]:
# Outliers
df_cont_transformed = df_transformed_features.select_dtypes(include="number")
df_winsor = df_cont_transformed.apply(lambda x: gestiona_outliers(x, clas='winsor'))

In [ ]:
# Coeficientes de asimetria luego de winsorizar
df_winsor.skew().sort_values(ascending=False)

In [ ]:
# Visualizar cambios
columna_a_graficar = 'MarketCap_Log1p' # indicar columna para el grafico
plot(df_winsor[columna_a_graficar])

In [ ]:
df_winsor.describe().T

## Concatenación final de columnas

In [ ]:
df_non_numeric_transformed = df_transformed_features.select_dtypes(exclude='number')
# Se unen variables contínuas transformadas y variables no numéricas
df_combined = pd.concat([df_non_numeric_transformed, df_winsor], axis=1)

# Unir con las columnas que fueron salteadas
df_final = pd.concat([df_passthrough, df_combined], axis=1)
df_final.info()

In [ ]:
# Guardar datos extraidos en fichero clean_data
df_final.to_parquet(f"{data_folder}/clean_data.parquet")
print(f"Fichero 'clean_data.parquet' guardado en la carpeta {data_folder}")